# 纯torch代码实现vllm中默认参数下的Embedding全过程


In [45]:
import torch
from torch import nn
import numpy as np

In [46]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从前一步得到的结果
token_ids = [9707, 9322, 11, 264, 1273, 11652, 151643]

vocab_size = 151669
# 查询得知

hidden_size = 1024
padding_idx = 151643

# 2. Embedding lookup


In [47]:
# nn.Embedding
embed_layer = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim = hidden_size,
    padding_idx=padding_idx,
)
embed_layer

Embedding(151669, 1024, padding_idx=151643)

In [48]:
token_ids = torch.tensor(token_ids, dtype=torch.long)
token_ids

tensor([  9707,   9322,     11,    264,   1273,  11652, 151643])

In [49]:
# 加载权重文件
from safetensors.torch import load_file

model_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/model.safetensors"

# 读取文件
state_dict = load_file(model_file)  # 返回 dict: key -> torch.Tensor

# 查看有哪些 key
print(list(state_dict.keys()))

['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_norm.weight', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.weight', 'layers.10.input_layernorm.weight', 'layers.10.mlp.down_proj.weight', 'layers.10.mlp.gate_proj.weight', 'layers.10.mlp.up_proj.weight', 'layers.10.post_attention_layernorm.weight', 'layers.10.

In [50]:
# 提取 embedding 权重
embedding_weights = state_dict['embed_tokens.weight']
embedding_weights.shape

torch.Size([151669, 1024])

In [51]:
# 将 safetensors 权重拷贝到 nn.Embedding
embed_layer.weight.data.copy_(embedding_weights)

tensor([[-0.0031,  0.0327, -0.0703,  ...,  0.0138, -0.0144,  0.0128],
        [ 0.0303,  0.0244, -0.0613,  ..., -0.0031, -0.0374,  0.0077],
        [ 0.0292,  0.0322, -0.0223,  ..., -0.0095,  0.0025,  0.0256],
        ...,
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026]])

In [52]:
# 使用 nn.Embedding 查表
embeddings = embed_layer(token_ids)  # shape: [7, 1024]
print(embeddings.shape)
print(embeddings)


torch.Size([7, 1024])
tensor([[ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
        [ 0.0031,  0.0045, -0.0206,  ..., -0.0312, -0.0231,  0.0123],
        [-0.0201,  0.0503, -0.0757,  ..., -0.0210,  0.0055, -0.0255],
        ...,
        [-0.0214, -0.0391, -0.0535,  ...,  0.0430, -0.0386, -0.0222],
        [ 0.0170, -0.0674,  0.0131,  ...,  0.0086,  0.0210, -0.0094],
        [-0.0043,  0.0435, -0.0334,  ..., -0.0039,  0.0449,  0.0320]],
       grad_fn=<EmbeddingBackward0>)


从 tokenizer 得到 token id 序列

根据 token id 在 embedding 矩阵中索引对应向量

为 batch 做 padding（下面讲 mask）
# 位置信息（Positional Encoding）——告诉模型“顺序”